In [ ]:
import torch
import numpy as np
import yaml
import open3d as o3d
import open3d.ml.torch as ml3d
import warnings

In [ ]:
print(torch.__version__)
print(torch.cuda.is_available())
print(np.__version__)
print(o3d.__version__)

2.2.2+cu121
True
1.26.4
0.19.0


In [ ]:
# poss_dataset.py
import numpy as np
import yaml
import open3d.ml.torch as ml3d

# POSS (17)
POSS_LABELS = {
    0:  "unlabeled",
    1:  "1 person",
    2:  "2+ person",
    3:  "rider",
    4:  "car",
    5:  "trunk",
    6:  "plants",
    7:  "traffic sign 1",
    8:  "traffic sign 2",
    9:  "traffic sign 3",
    10: "pole",
    11: "trashcan",
    12: "building",
    13: "cone/stone",
    14: "fence",
    15: "bike",
    16: "ground",
}

class POSSDataset(ml3d.datasets.SemanticKITTI):

    def __init__(
        self,
        dataset_path: str,
        poss_yaml_path: str, # <--- poss.yaml
        name: str = "poss",
        cache_dir: str = "./logs/cache_poss",
        use_cache: bool = False,
        class_weights=None,
        ignored_label_inds=None,
        test_result_folder=None,
        test_split=None,
        training_split=None,
        validation_split=None,
        all_split=None,
        **kwargs,
    ):
        super().__init__(dataset_path=dataset_path,
                         name=name,
                         cache_dir=cache_dir,
                         use_cache=use_cache,
                         class_weights=class_weights,
                         ignored_label_inds=ignored_label_inds or [],
                         test_result_folder=test_result_folder,
                         training_split=training_split or ["00","01","02","03"],
                         validation_split=validation_split or ["04"],
                         test_split=test_split or ["05"],
                         all_split=all_split or ["00","01","02","03","04","05"],
                         **kwargs)

        self._poss_yaml_path = poss_yaml_path
        with open(self._poss_yaml_path, "r") as f:
            DATA = yaml.safe_load(f)

        self.label_to_names = self.get_label_to_names()
        self.num_classes = len(self.label_to_names)

        # learning_map_inv: train_id -> raw_id
        remap_dict_inv = DATA["learning_map_inv"]
        max_key = max(remap_dict_inv.keys()) if remap_dict_inv else 0
        remap_lut = np.zeros((max_key + 100), dtype=np.int32)
        remap_lut[list(remap_dict_inv.keys())] = list(remap_dict_inv.values())

        # learning_map: raw_id -> train_id
        remap_dict = DATA["learning_map"]
        max_key_val = max(remap_dict.keys()) if remap_dict else 0
        remap_lut_val = np.zeros((max_key_val + 100), dtype=np.int32)
        remap_lut_val[list(remap_dict.keys())] = list(remap_dict.values())

        self.remap_lut = remap_lut
        self.remap_lut_val = remap_lut_val

    @staticmethod
    def get_label_to_names():
        return dict(POSS_LABELS)


In [ ]:
with open("randlanet_semantickitti.yml", 'r') as f:
    cfg = yaml.safe_load(f)

In [ ]:
ds_cfg = cfg['dataset']
ds = POSSDataset(
    dataset_path=ds_cfg['dataset_path'],
    poss_yaml_path="randlanet_semantickitti.yml",
    name=ds_cfg['name'],
    cache_dir=ds_cfg['cache_dir'],
    use_cache=ds_cfg['use_cache'],
    class_weights=ds_cfg['class_weights'],
    ignored_label_inds=[],
    test_result_folder=ds_cfg['test_result_folder'],
    test_split=ds_cfg['test_split'],
    training_split=ds_cfg['training_split'],
    validation_split=ds_cfg['validation_split'],
    all_split=ds_cfg['all_split'],
    sampler=ds_cfg['sampler'],
)

In [ ]:
model_cfg = cfg['model']
model = ml3d.models.RandLANet(
    name=model_cfg['name'],
    batcher=model_cfg['batcher'],
    num_neighbors=model_cfg['num_neighbors'],
    num_layers=model_cfg['num_layers'],
    num_points=model_cfg['num_points'],
    num_classes=model_cfg['num_classes'],
    ignored_label_inds=model_cfg['ignored_label_inds'],
    sub_sampling_ratio=model_cfg['sub_sampling_ratio'],
    in_channels=model_cfg['in_channels'],
    dim_features=model_cfg['dim_features'],
    dim_output=model_cfg['dim_output'],
    grid_size=model_cfg['grid_size'],
    augment=model_cfg['augment'],
)

In [ ]:
pipe_cfg = cfg['pipeline']
pipeline = ml3d.pipelines.SemanticSegmentation(
    model=model,
    dataset=ds,
    name=pipe_cfg['name'],
    optimizer=pipe_cfg['optimizer'],
    batch_size=pipe_cfg['batch_size'],
    main_log_dir=pipe_cfg['main_log_dir'],
    max_epoch=1,
    save_ckpt_freq=pipe_cfg['save_ckpt_freq'],
    scheduler_gamma=pipe_cfg['scheduler_gamma'],
    test_batch_size=pipe_cfg['test_batch_size'],
    train_sum_dir=pipe_cfg['train_sum_dir'],
    val_batch_size=pipe_cfg['val_batch_size'],
    summary=pipe_cfg['summary'],
);

In [ ]:
print("Train sample shape:", ds.get_split("training").get_data(0)['point'].shape)
print("Train sample feat:", ds.get_split("training").get_data(0)['feat'].shape)
print("Val sample shape:", ds.get_split("validation").get_data(0)['point'].shape)
print("Val sample feat:", ds.get_split("validation").get_data(0)['feat'].shape)
print("Test sample shape:", ds.get_split("test").get_data(0)['point'].shape)
print("Test sample feat:", ds.get_split("test").get_data(0)['feat'].shape)

Train sample shape: (66658, 3)
Train sample feat: (66658, 1)
Val sample shape: (66303, 3)
Val sample feat: (66303, 1)
Test sample shape: (66534, 3)
Test sample feat: (66534, 1)


In [ ]:
split = ds.get_split("test")
sample = split.get_data(0)
points = sample["point"].astype(np.float32)
feats = sample["feat"].astype(np.float32)
label = sample["label"].astype(np.int32)

data_infer = {
    "point": points,
    "feat": feats,
    "label": label,
    }
res = pipeline.run_inference(data_infer)
pred = np.asarray(res["predict_labels"]).astype(np.int32)

test 0/1:  95%|█████████▌| 59871/62751 [00:00<00:00, 156787.53it/s]

In [ ]:
res

{'predict_labels': array([0, 0, 0, ..., 0, 0, 0]),
 'predict_scores': array([[0.01974 , 0.013985, 0.01032 , ..., 0.01758 , 0.01434 , 0.0176  ],
        [0.01967 , 0.01399 , 0.01037 , ..., 0.01755 , 0.014336, 0.0176  ],
        [0.01958 , 0.014   , 0.01043 , ..., 0.01752 , 0.014336, 0.01761 ],
        ...,
        [0.03232 , 0.02472 , 0.0211  , ..., 0.02705 , 0.02759 , 0.02847 ],
        [0.03238 , 0.02469 , 0.0211  , ..., 0.02708 , 0.02762 , 0.02846 ],
        [0.03238 , 0.02469 , 0.0211  , ..., 0.02708 , 0.02762 , 0.02846 ]],
       dtype=float16)}

In [ ]:
names = ds.get_label_to_names()
num_classes = int(max(names.keys())) + 1
rng = np.random.default_rng(0)
palette = rng.random((num_classes, 3))

pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(points))
pcd.colors = o3d.utility.Vector3dVector(palette[pred])
o3d.visualization.draw_geometries([pcd], window_name="RandLA-Net predictions")

In [ ]:
train_split = ds.get_split("training")
print(train_split.get_data(0)['point'].shape)

val_split = ds.get_split("validation")
print(val_split.get_data(0)['point'].shape)

test_split = ds.get_split("test")
print(test_split.get_data(0)['point'].shape)

(66658, 3)
(66303, 3)
(66534, 3)


In [ ]:
result = pipeline.run_train()


preprocess: 100%|██████████| 1988/1988 [02:53<00:00, 11.45it/s]

preprocess: 100%|██████████| 500/500 [00:37<00:00, 13.17it/s]

training:   0%|          | 0/497 [00:00<?, ?it/s]/home/artniz/3d/homework_11/venv/lib/python3.12/site-packages/open3d/_ml3d/torch/dataloaders/default_batcher.py:48: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  storage = elem.storage()._new_shared(numel)
/home/artniz/3d/homework_11/venv/lib/python3.12/site-packages/open3d/_ml3d/torch/dataloaders/default_batcher.py:48: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() in

In [ ]:
pipeline.run_test()


preprocess: 100%|██████████| 500/500 [01:14<00:00,  6.74it/s]

test 0/1: 100%|██████████| 62751/62751 [15:14<00:00, 68.61it/s]    

test 1/500: 100%|██████████| 62679/62679 [00:01<00:00, 33586.65it/s]

test 3/500: 100%|██████████| 62813/62813 [00:02<00:00, 30868.62it/s] 

test 5/500: 100%|██████████| 62785/62785 [00:02<00:00, 30172.94it/s]

test 7/500: 100%|██████████| 62850/62850 [00:02<00:00, 30067.79it/s]

test 9/500: 100%|██████████| 62946/62946 [00:01<00:00, 31883.14it/s]

test 11/500: 100%|██████████| 62852/62852 [00:02<00:00, 26384.62it/s]

test 13/500: 100%|██████████| 62930/62930 [00:02<00:00, 25409.88it/s]

test 15/500: 100%|██████████| 62931/62931 [00:02<00:00, 30085.56it/s]

test 17/500: 100%|██████████| 62986/62986 [00:02<00:00, 29324.71it/s] 

test 19/500: 100%|██████████| 62763/62763 [00:02<00:00, 29761.44it/s]

test 21/500: 100%|██████████| 62563/62563 [00:02<00:00, 30174.74it/s]

test 23/500: 100%|██████████| 62606/62606 [00:01<00:00, 34949.59it/s]

test 25/500: 100%|

In [ ]:
# Метрики
print(f"Overall Train Accuracy : {pipeline.metric_train.acc()[-1]}, mIoU : {pipeline.metric_train.iou()[-1]}")
print(f"Overall Validation Accuracy : {pipeline.metric_val.acc()[-1]}, mIoU : {pipeline.metric_val.iou()[-1]}")
print(f"Overall Testing Accuracy : {pipeline.metric_test.acc()[-1]}, mIoU : {pipeline.metric_test.iou()[-1]}")

Overall Train Accuracy : 0.29284780597852156, mIoU : 0.1859500796006963
Overall Validation Accuracy : 0.297428888291863, mIoU : 0.23684304523164443
Overall Testing Accuracy : 0.2864190771752353, mIoU : 0.22881094047654033
